# Введение в MapReduce модель на Python


In [1]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [2]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [3]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [4]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [5]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [6]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [7]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [8]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [9]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [10]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [11]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [12]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [13]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [14]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [15]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(1.4083024511366138)),
 (1, np.float64(1.4083024511366138)),
 (2, np.float64(1.4083024511366138)),
 (3, np.float64(1.4083024511366138)),
 (4, np.float64(1.4083024511366138))]

## Inverted index

In [16]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('what', ['0', '1']),
 ('is', ['0', '1', '2']),
 ('it', ['0', '1', '2']),
 ('a', ['2']),
 ('banana', ['2'])]

## WordCount

In [17]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [18]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [19]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('a', 2)]),
 (1, [('banana', 2), ('is', 18), ('it', 18), ('what', 10)])]

## TeraSort

In [20]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.02788439466179149)),
   (None, np.float64(0.04353084482978942)),
   (None, np.float64(0.04661745683048579)),
   (None, np.float64(0.08604219354254616)),
   (None, np.float64(0.08656912643227943)),
   (None, np.float64(0.12915711789764583)),
   (None, np.float64(0.13691903992767973)),
   (None, np.float64(0.2510117204933542)),
   (None, np.float64(0.2868899860553725)),
   (None, np.float64(0.36262052268693057)),
   (None, np.float64(0.4045795250240556)),
   (None, np.float64(0.4873514851489448))]),
 (1,
  [(None, np.float64(0.5474853495493509)),
   (None, np.float64(0.5523472429450744)),
   (None, np.float64(0.6111491771625795)),
   (None, np.float64(0.6240997453518179)),
   (None, np.float64(0.6301344887014969)),
   (None, np.float64(0.6340469002460664)),
   (None, np.float64(0.638532575875415)),
   (None, np.float64(0.7053106451423368)),
   (None, np.float64(0.7359338012413064)),
   (None, np.float64(0.7553937966914077)),
   (None, np.float64(0.787395110552

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [21]:
input = [1, 8, 9, 3, 14, 8, 1, 20]

def RECORDREADER():
  for value in input:
    yield (0, value)

def MAP(id, value):
  yield(id, value)

def REDUCE(id, arr):
  yield(id, max(arr))

list(MapReduce(RECORDREADER, MAP, REDUCE))

[(0, 20)]

### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [22]:
input = [1, 8, 9, 3, 14, 8, 1, 20]

def RECORDREADER():
  for value in input:
    yield (0, value)

def MAP(id, value):
  yield(id, value)

def REDUCE(id, arr):
  yield(id, sum(arr)/len(arr))

list(MapReduce(RECORDREADER, MAP, REDUCE))

[(0, 8.0)]

### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [23]:
from itertools import groupby

data = [
    User(id=0, age=18, gender='male', social_contacts=50),
    User(id=1, age=18, gender='female', social_contacts=300),
    User(id=2, age=25, gender='male', social_contacts=120),
    User(id=3, age=35, gender='female', social_contacts=450),
    User(id=4, age=35, gender='male', social_contacts=200),
    User(id=5, age=45, gender='female', social_contacts=600),
]

def RECORDREADER(key:str):
  for user in data:
    yield (getattr(user, key), user)

def groupByKey(iterable):
    sorted_pairs = sorted(iterable, key=lambda x: x[0])
    return [(k, [v for _, v in g]) for k, g in groupby(sorted_pairs, key=lambda x: x[0])]

print(list(groupByKey(flatten(map(lambda x: MAP(*x), RECORDREADER("age"))))))
print(list(groupByKey(flatten(map(lambda x: MAP(*x), RECORDREADER("gender"))))))

[(18, [User(id=0, age=18, social_contacts=50, gender='male'), User(id=1, age=18, social_contacts=300, gender='female')]), (25, [User(id=2, age=25, social_contacts=120, gender='male')]), (35, [User(id=3, age=35, social_contacts=450, gender='female'), User(id=4, age=35, social_contacts=200, gender='male')]), (45, [User(id=5, age=45, social_contacts=600, gender='female')])]
[('female', [User(id=1, age=18, social_contacts=300, gender='female'), User(id=3, age=35, social_contacts=450, gender='female'), User(id=5, age=45, social_contacts=600, gender='female')]), ('male', [User(id=0, age=18, social_contacts=50, gender='male'), User(id=2, age=25, social_contacts=120, gender='male'), User(id=4, age=35, social_contacts=200, gender='male')])]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [24]:
data = [
    User(id=0, age=18, gender='male', social_contacts=50),
    User(id=1, age=18, gender='female', social_contacts=300),
    User(id=2, age=25, gender='male', social_contacts=120),
    User(id=3, age=35, gender='female', social_contacts=450),
    User(id=4, age=35, gender='male', social_contacts=200),
    User(id=4, age=35, gender='male', social_contacts=200),
]

def RECORDREADER():
    for i, user in enumerate(data):
        yield (i, user)

def MAP(key, user):
    yield (user, 1)

def REDUCE(user, values):
    yield user
    
list(MapReduce(RECORDREADER, MAP, REDUCE))

[User(id=0, age=18, social_contacts=50, gender='male'),
 User(id=1, age=18, social_contacts=300, gender='female'),
 User(id=2, age=25, social_contacts=120, gender='male'),
 User(id=3, age=35, social_contacts=450, gender='female'),
 User(id=4, age=35, social_contacts=200, gender='male')]

# Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [25]:
data = [
    User(id=0, age=18, gender='male', social_contacts=50),
    User(id=1, age=18, gender='female', social_contacts=300),
    User(id=2, age=25, gender='male', social_contacts=120),
    User(id=3, age=35, gender='female', social_contacts=450),
    User(id=4, age=35, gender='male', social_contacts=200),
    User(id=5, age=45, gender='female', social_contacts=600),
]

def MAP(userId, user):
  if user.age < 35:
    yield (userId, user)

def REDUCE(userId, user):
  yield (userId, user)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, [User(id=0, age=18, social_contacts=50, gender='male')]),
 (1, [User(id=1, age=18, social_contacts=300, gender='female')]),
 (2, [User(id=2, age=25, social_contacts=120, gender='male')])]

### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [26]:
class UserProjection(NamedTuple):
  id: int
  age: int

def MAP(userId, user):
  newUser = UserProjection(user.id, user.age)
  yield(newUser, newUser)

def REDUCE(key, usersList):
  yield (key, key)

list(MapReduce(RECORDREADER, MAP, REDUCE))

[(UserProjection(id=0, age=18), UserProjection(id=0, age=18)),
 (UserProjection(id=1, age=18), UserProjection(id=1, age=18)),
 (UserProjection(id=2, age=25), UserProjection(id=2, age=25)),
 (UserProjection(id=3, age=35), UserProjection(id=3, age=35)),
 (UserProjection(id=4, age=35), UserProjection(id=4, age=35)),
 (UserProjection(id=5, age=45), UserProjection(id=5, age=45))]

### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [27]:
data_1 = [
    User(id=10, age=28, gender='female', social_contacts=150),
    User(id=11, age=42, gender='male', social_contacts=320),
    User(id=12, age=35, gender='female', social_contacts=890),
    User(id=13, age=19, gender='male', social_contacts=45),
]

data_2 = [
    User(id=12, age=35, gender='female', social_contacts=890),
    User(id=14, age=51, gender='female', social_contacts=1200),
    User(id=15, age=28, gender='male', social_contacts=275),
    User(id=13, age=19, gender='male', social_contacts=45),
    User(id=16, age=63, gender='female', social_contacts=560),
]

def RECORDREADER():
  for i, user in enumerate(data_1):
        yield (i, user)
  for i, user in enumerate(data_2):
      yield (i, user)

def MAP(userId, user):
  yield(user, user)

def REDUCE(key, usersList):
  yield (key, key)


list(MapReduce(RECORDREADER, MAP, REDUCE))

[(User(id=10, age=28, social_contacts=150, gender='female'),
  User(id=10, age=28, social_contacts=150, gender='female')),
 (User(id=11, age=42, social_contacts=320, gender='male'),
  User(id=11, age=42, social_contacts=320, gender='male')),
 (User(id=12, age=35, social_contacts=890, gender='female'),
  User(id=12, age=35, social_contacts=890, gender='female')),
 (User(id=13, age=19, social_contacts=45, gender='male'),
  User(id=13, age=19, social_contacts=45, gender='male')),
 (User(id=14, age=51, social_contacts=1200, gender='female'),
  User(id=14, age=51, social_contacts=1200, gender='female')),
 (User(id=15, age=28, social_contacts=275, gender='male'),
  User(id=15, age=28, social_contacts=275, gender='male')),
 (User(id=16, age=63, social_contacts=560, gender='female'),
  User(id=16, age=63, social_contacts=560, gender='female'))]

### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [28]:
def MAP(userId, user):
  yield(user, user)

def REDUCE(key, usersList):
  if len(usersList) == 2:
    yield (key, key)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(User(id=12, age=35, social_contacts=890, gender='female'),
  User(id=12, age=35, social_contacts=890, gender='female')),
 (User(id=13, age=19, social_contacts=45, gender='male'),
  User(id=13, age=19, social_contacts=45, gender='male'))]

### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [29]:
def RECORDREADER():
  for user in data_1:
        yield (0, user)
  for user in data_2:
      yield (1, user)

def MAP(listId, user):
  yield(user, listId)

def REDUCE(key, setList):
  if len(setList)==1 and setList[0] == 0:
    yield (key, key)

list(MapReduce(RECORDREADER, MAP, REDUCE))

[(User(id=10, age=28, social_contacts=150, gender='female'),
  User(id=10, age=28, social_contacts=150, gender='female')),
 (User(id=11, age=42, social_contacts=320, gender='male'),
  User(id=11, age=42, social_contacts=320, gender='male'))]

### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [30]:
R = [
    (0, 25),
    (1, 33),
    (2, 25),
    (3, 55),
]

S = [
    (25, 240),
    (25, 500),
    (33, 800),
    (40, 150),
]

def RECORDREADER():
    for a, b in R:
        yield ('R', (a, b))
    for b, c in S:
        yield ('S', (b, c))

def MAP(source, tuple_data):
    if source == 'R':
        a, b = tuple_data
        yield (b, ('R', a))
    else:
        b, c = tuple_data
        yield (b, ('S', c))

def REDUCE(b, values):
    r_values = [a for (src, a) in values if src == 'R']
    s_values = [c for (src, c) in values if src == 'S']
    
    for a in r_values:
        for c in s_values:
            yield (a, b, c)

list(MapReduce(RECORDREADER, MAP, REDUCE))

[(0, 25, 240), (0, 25, 500), (2, 25, 240), (2, 25, 500), (1, 33, 800)]

### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [31]:
data = [
    User(id=0, age=18, gender='male', social_contacts=50),
    User(id=1, age=18, gender='female', social_contacts=300),
    User(id=2, age=25, gender='male', social_contacts=120),
    User(id=3, age=35, gender='female', social_contacts=450),
    User(id=4, age=35, gender='male', social_contacts=200),
    User(id=5, age=45, gender='female', social_contacts=600),
]

def RECORDREADER():
    for i, user in enumerate(data):
        yield (i, user)
        
def MAP(UserId, user):
    yield (user.gender, user.age)
    
def REDUCE(gender, age):
    yield (gender, sum(age)/len(age))

list(MapReduce(RECORDREADER, MAP, REDUCE))

[('male', 26.0), ('female', 32.666666666666664)]

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [32]:
data_matrix = np.array([
    [2, 4, 6, 8],
    [10, 12, 14, 16],
    [18, 20, 22, 24],
    [26, 28, 30, 32],
    [34, 36, 38, 40]
], dtype=float)

data_vector = np.array([5, 15, 25, 35], dtype=float)

maps = 2
reducers = 2
stripe_count = maps

def INPUTFORMAT():
    global maps, stripe_count, data_matrix, data_vector

    columns_per_stripe = data_matrix.shape[1] // stripe_count

    for stripe_idx in range(stripe_count):
        col_start = stripe_idx * columns_per_stripe
        col_end = (stripe_idx + 1) * columns_per_stripe

        v_segment = data_vector[col_start:col_end]
        m_segment = data_matrix[:, col_start:col_end]

        def RECORDREADER(mat_part=m_segment, vec_part=v_segment, offset=col_start):
            for row_idx in range(mat_part.shape[0]):
                for col_local in range(mat_part.shape[1]):
                    col_global = offset + col_local
                    yield ((row_idx, col_global), (mat_part[row_idx, col_local], vec_part[col_local]))

        yield RECORDREADER()

def MAP(coords, vals):
    row_idx, col_idx = coords
    m_val, v_val = vals
    yield (row_idx, m_val * v_val)

def REDUCE(row_key, partial_products):
    yield (row_key, sum(partial_products))


partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output


20 key-value pairs were sent over a network.


[(0,
  [(0, np.float64(500.0)), (2, np.float64(1780.0)), (4, np.float64(3060.0))]),
 (1, [(1, np.float64(1140.0)), (3, np.float64(2420.0))])]

In [33]:
np.matmul(data_matrix, data_vector)

array([ 500., 1140., 1780., 2420., 3060.])

## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [34]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [35]:
import numpy as np
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J) # it is legal to access this from RECORDREADER, MAP, REDUCE
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j,k), big_mat[j,k])


def MAP(k1, v1):
    (j, k) = k1
    w = v1
    for i in range(I):
        yield ((i, k), small_mat[i, j] * w)


def REDUCE(key, values):
    (i, k) = key
    yield (i, k), sum(values)

Проверьте своё решение

In [36]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [37]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [38]:
I = 3
J = 4
K = 5 * 10

small_mat = np.random.rand(I, J) # it is legal to access this from RECORDREADER, MAP, REDUCE
big_mat = np.random.rand(J, K)

def RECORDREADER():
  for i in range(small_mat.shape[0]):
    for j in range(big_mat.shape[0]):
      for k in range(big_mat.shape[1]):
        yield (((i, j), small_mat[i, j]),((j, k), big_mat[j, k]))

def MAP(list_1, list_2):
  (k1, v1) = list_1
  (k2, v2) = list_2
  yield ((k1[0], k2[1]), v1 * v2)

def REDUCE(key, values):
  (i, k) = key
  yield ((i, k), sum(values))

In [39]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [40]:
maps = 2
reducers = 3

small_mat = np.random.rand(I, J)
big_mat = np.random.rand(J, K)

def INPUTFORMAT():
  def RECORDREADER(key, split):
    mat = []
    for i in range(split.shape[0]):
      for j in range(split.shape[1]):
        mat.append(((key, i, j), split[i, j]))
    return mat

  yield RECORDREADER('S', small_mat)
  yield RECORDREADER('B', big_mat)

def MAP(k, v):
  (mat, i, j) = k
  w = v
  if mat == 'S':
    yield (j, (mat, i, w))
  else:
    yield (i, (mat, j, w))

def REDUCE(_, values):
  small = [v for v in values if v[0] == 'S']
  big = [v for v in values if v[0] == 'B']
  for s in small:
    for b in big:
      yield ((s[1], b[1]), s[2] * b[2])

def INPUT_MUL():
  for j in joined:
    yield j[1]

def MAP_MUL(k1, v1):
  yield (k1, v1)

def REDUCE_MUL(key, values):
  res_val = 0
  for v in values:
    res_val += v
  yield (key, res_val)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
joined = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]

mul_output = MapReduceDistributed(INPUT_MUL, MAP_MUL, REDUCE_MUL, COMBINER=None)
pre_result = [(partition_id, list(partition)) for (partition_id, partition) in mul_output]

solution = []
for p in pre_result:
    for v in p[1]:
        solution.append(v)

212 key-value pairs were sent over a network.
600 key-value pairs were sent over a network.


In [41]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
np.allclose(reference_solution, asmatrix(solution)) # should return true

True

Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [42]:
small_mat = np.random.rand(I, J)
big_mat = np.random.rand(J, K)

def INPUTFORMAT():
    global maps
    def RECORDREADER(key, split):
        mat = []
        for i in range(split.shape[0]):
            for j in range(split.shape[1]):
                mat.append(((key, i, j), split[i, j]))
        return mat

    small = RECORDREADER('S', small_mat)
    split_size =  int(np.ceil(len(small)/maps))
    for i in range(0, len(small), split_size):
        yield small[i:i+split_size]

    big = RECORDREADER('B', big_mat)
    split_size =  int(np.ceil(len(big)/maps))
    for i in range(0, len(big), split_size):
      yield big[i:i+split_size]

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
joined = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]

mul_output = MapReduceDistributed(INPUT_MUL, MAP_MUL, REDUCE_MUL, COMBINER=None)
pre_result = [(partition_id, list(partition)) for (partition_id, partition) in mul_output]

solution = []
for p in pre_result:
    for v in p[1]:
      solution.append(v)

212 key-value pairs were sent over a network.
600 key-value pairs were sent over a network.


In [43]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
np.allclose(reference_solution, asmatrix(solution)) # should return true

True

**Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?**

Да, решение останется работоспособным.

Это возможно благодаря тому, что в рассматриваемой архитектуре каждый элемент матрицы передаётся в явном виде вместе со своими координатами, а не определяется неявно через его позицию в потоке.

Если RECORDREADER выдаёт лишь случайную часть элементов, пропущенные данные просто не участвуют в вычислениях. С математической точки зрения это равносильно тому, что отсутствующие элементы равны нулю. Поскольку умножение любого числа на ноль даёт ноль, а сложение с нулём не меняет сумму, итоговый результат для обработанных элементов остаётся точным.

Такой подход является стандартной практикой в распределённых вычислениях для работы с разреженными матрицами, где хранение и обработка только ненулевых значений позволяет существенно экономить ресурсы.

Важно: итоговые значения будут соответствовать произведению разреженных матриц, а не полных. Это не является ошибкой, а представляет собой корректное поведение алгоритма для данного типа данных.